# TA-Lib Indicator Tester (One-by-One)

Interactive notebook to load your OHLC CSV and test **each TA-Lib indicator** individually.

**Setup** (run once in terminal if needed):
```bash
pip install -e ".[dev]"
pip install jupyter ipywidgets
```

**How to use**
1. Run **Setup** and **Load data** cells.
2. Pick an indicator from the dropdown (or set `INDICATOR` manually).
3. Run **Test one indicator** to validate + plot.
4. Use **Previous / Next** buttons to step through all indicators one by one.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Project root (notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from indicator_testing.data_loader import describe_ohlc, load_ohlc
from indicator_testing.plotter import plot_indicator
from indicator_testing.registry import IndicatorRegistry
from indicator_testing.runner import run_indicator

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("Tip: pip install ipywidgets for dropdown + prev/next buttons")

print(f"Project root: {PROJECT_ROOT}")

## Configuration

Edit the paths and options below, then re-run this cell.

In [ ]:
# --- edit these ---
CSV_PATH = PROJECT_ROOT / "questdb-query-1781940224994.csv"  # your QuestDB export
# CSV_PATH = PROJECT_ROOT / "data" / "sample_monthly.csv"

RESAMPLE = "daily"   # "none" | "daily" | "monthly" | "weekly"
SYMBOL = None        # e.g. "NIFTY" if multiple symbols in CSV

# Manual override (used when ipywidgets not installed)
INDICATOR = "RSI"

# Optional param overrides for the current indicator
PARAMS = {}  # e.g. {"timeperiod": 14}

In [ ]:
resample_arg = None if RESAMPLE == "none" else RESAMPLE
df = load_ohlc(CSV_PATH, resample=resample_arg, symbol=SYMBOL, warn_short=False)
info = describe_ohlc(df)

registry = IndicatorRegistry()
all_indicators = [i.name for i in registry.iter_ordered()]
groups = registry.groups()

print(f"Loaded {info['bars']} bars | {info['inferred_frequency']}")
print(f"Range: {info['start']} -> {info['end']}")
print(f"Close: {info['close_min']:.2f} - {info['close_max']:.2f} | Volume: {info['has_volume']}")
print(f"TA-Lib indicators available: {len(all_indicators)}")
print()
for gname, names in groups.items():
    print(f"  {gname}: {len(names)}")

df.tail(3)

## Pick an indicator

Use the dropdown and **Prev / Next** buttons, or set `INDICATOR = "MACD"` in the config cell.

In [ ]:
current_index = all_indicators.index(INDICATOR.upper()) if INDICATOR.upper() in all_indicators else 0

if HAS_WIDGETS:
    indicator_dropdown = widgets.Dropdown(
        options=[(f"{n} ({registry.get(n).group})", n) for n in all_indicators],
        value=all_indicators[current_index],
        description="Indicator:",
        layout=widgets.Layout(width="500px"),
    )
    btn_prev = widgets.Button(description="<< Prev", button_style="info")
    btn_next = widgets.Button(description="Next >>", button_style="info")
    out_pick = widgets.Output()

    def _sync_index(change=None):
        global current_index
        current_index = all_indicators.index(indicator_dropdown.value)

    def _go_prev(_):
        global current_index
        current_index = (current_index - 1) % len(all_indicators)
        indicator_dropdown.value = all_indicators[current_index]

    def _go_next(_):
        global current_index
        current_index = (current_index + 1) % len(all_indicators)
        indicator_dropdown.value = all_indicators[current_index]

    indicator_dropdown.observe(_sync_index, names="value")
    btn_prev.on_click(_go_prev)
    btn_next.on_click(_go_next)

    display(widgets.HBox([btn_prev, indicator_dropdown, btn_next]))
else:
    print(f"Current INDICATOR = {INDICATOR}")
    print("Available (first 20):", all_indicators[:20])

## Test one indicator

Runs TA-Lib, validates output, shows a sample table, and plots a chart.

In [ ]:
def get_selected_indicator():
    if HAS_WIDGETS:
        return indicator_dropdown.value
    return INDICATOR.upper()


def show_validation(result):
    info = registry.get(result.name)
    print(f"{'='*60}")
    print(f"{result.name}  |  {result.group}")
    print(f"Status: {result.status}  |  Warmup bars: {result.warmup_bars}")
    print(f"Inputs: {info.input_names}  |  Outputs: {info.output_names}")
    print(f"Params: {result.params_used}")
    if result.message and result.status != "success":
        print(f"Message: {result.message}")
    if result.validation:
        tag = "PASS" if result.validation.passed else "FAIL"
        print(f"Validation: {tag}")
        for c in result.validation.checks:
            mark = "OK" if c.passed else "X"
            print(f"  [{mark}] {c.name}: {c.message}")
    print(f"{'='*60}")


def result_to_frame(result, n_tail=15):
    if result.status != "success" or not result.outputs:
        return pd.DataFrame()
    out = pd.DataFrame({"close": df["close"]})
    for k, arr in result.outputs.items():
        out[k] = arr
    out.index = df.index
    warmup = result.warmup_bars or 0
    out["_warmup"] = [i < warmup for i in range(len(out))]
    return out.tail(n_tail)


def plot_inline(result):
    if result.status != "success" or not result.outputs:
        print("Nothing to plot.")
        return
    chart_path = PROJECT_ROOT / "outputs" / "charts" / "notebook_preview.png"
    chart_path.parent.mkdir(parents=True, exist_ok=True)
    plot_indicator(df, result, chart_path, show=False)
    img = plt.imread(chart_path)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(result.name)
    plt.show()


name = get_selected_indicator()
result = run_indicator(name, df, registry, params=PARAMS or None)
show_validation(result)
display(result_to_frame(result))
plot_inline(result)

## Browse all indicators (one by one)

Run this cell to step through **every** indicator sequentially.  
Set `MAX_TO_SHOW = 5` to limit how many are displayed per run, or `None` for all.

In [ ]:
MAX_TO_SHOW = None  # e.g. 10 to preview first 10 only
SHOW_PLOT_EACH = False  # True = plot every indicator (slow for 158 indicators)

rows = []
to_run = all_indicators[:MAX_TO_SHOW] if MAX_TO_SHOW else all_indicators

for i, name in enumerate(to_run, start=1):
    r = run_indicator(name, df, registry)
    valid = r.validation.passed if r.validation else None
    rows.append({
        "#": i,
        "indicator": r.name,
        "group": r.group,
        "status": r.status,
        "warmup": r.warmup_bars,
        "valid": valid,
        "message": r.message if r.status != "success" else "",
    })
    if SHOW_PLOT_EACH and r.status == "success":
        print(f"[{i}/{len(to_run)}] {name}")
        plot_inline(r)

summary = pd.DataFrame(rows)
print(f"Done: {len(summary)} indicators")
print(summary["status"].value_counts().to_string())
summary

## Filter summary

Show only failures, skips, or a specific group from the batch run above.

In [ ]:
# Re-run previous cell first if 'summary' is not defined
if "summary" in dir():
    display(summary[summary["status"] != "success"])
    print()
    display(summary.groupby("group")["status"].value_counts())
else:
    print("Run the 'Browse all indicators' cell first.")

## Quick reference — indicators by group

In [ ]:
for gname, names in groups.items():
    print(f"\n=== {gname} ({len(names)}) ===")
    for n in names:
        info = registry.get(n)
        vol = " [vol]" if info.requires_volume else ""
        print(f"  {n:<18} in={info.input_names} out={info.output_names}{vol}")